In [5]:
import pandas as pd
import torch
from transformers import pipeline

/Users/atyantjain/Desktop/Personal Projects/Practice/.llm/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/atyantjain/Desktop/Personal Projects/Practice/.llm/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
data ={
    "customer_feedback": [
        "The app is easy to use but sometimes slow.",
        "I love the new AI assistant feature.",
        "The dashboard is confusing and needs better design.",
        "Customer support was helpful and quick.",
        "The recommendation system feels very accurate."
    ]
}

In [7]:
df = pd.DataFrame(data)

# -----------------------------
# 2. Basic preprocessing
# -----------------------------
df["clean_text"] = df["customer_feedback"].str.lower().str.strip()

print("Cleaned Dataset:")
print(df)

Cleaned Dataset:
                                   customer_feedback  \
0         The app is easy to use but sometimes slow.   
1               I love the new AI assistant feature.   
2  The dashboard is confusing and needs better de...   
3            Customer support was helpful and quick.   
4     The recommendation system feels very accurate.   

                                          clean_text  
0         the app is easy to use but sometimes slow.  
1               i love the new ai assistant feature.  
2  the dashboard is confusing and needs better de...  
3            customer support was helpful and quick.  
4     the recommendation system feels very accurate.  


In [8]:
generator = pipeline("text-generation",model="openai/whisper-large-v3",device="mps" if torch.backends.mps.is_available() else -1)

Device set to use mps


In [9]:
def generate_insight(text):
    prompt = f"""  Summarise the following customer feedback and suggest one improvement:

    Feedback: {text}

    Summary and suggestion:
    """
    output = generator (
        prompt,
        max_new_tokens=40,
        num_return_sequences=1,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
        repetition_penalty=1.2,
        pad_token_id=50256

    )
    result=output[0]["generated_text"]
    return result.replace(prompt, "").strip()

In [10]:
df["ai_genrated_response"]=df["clean_text"].apply(generate_insight)

In [11]:
print(df)

                                   customer_feedback  \
0         The app is easy to use but sometimes slow.   
1               I love the new AI assistant feature.   
2  The dashboard is confusing and needs better de...   
3            Customer support was helpful and quick.   
4     The recommendation system feels very accurate.   

                                          clean_text  \
0         the app is easy to use but sometimes slow.   
1               i love the new ai assistant feature.   
2  the dashboard is confusing and needs better de...   
3            customer support was helpful and quick.   
4     the recommendation system feels very accurate.   

                                ai_genrated_response  
0                                                ...  
1                     .. Kana, S.P. P.T.S.EA soyder?  
2  �� w-stu'elica,yentrynkdjnrkfhkdhkd'stripte He...  
3  �� l아뵑 Tcha, YI HASTARASI'SICTiqemprinentibech...  
4  ��ibertypzalponopepotensi'kosyantap aqeyesкaka..

In [12]:
for index, row in df.iterrows():
    print("\n-----------------------------")
    print("Original Feedback:")
    print(row["customer_feedback"])

    print("\nAI Generated Insight:")
    print(row["ai_genrated_response"])


-----------------------------
Original Feedback:
The app is easy to use but sometimes slow.

AI Generated Insight:
...

-----------------------------
Original Feedback:
I love the new AI assistant feature.

AI Generated Insight:
.. Kana, S.P. P.T.S.EA soyder?

-----------------------------
Original Feedback:
The dashboard is confusing and needs better design.

AI Generated Insight:
�� w-stu'elica,yentrynkdjnrkfhkdhkd'stripte He CrileGOTRN

-----------------------------
Original Feedback:
Customer support was helpful and quick.

AI Generated Insight:
�� l아뵑 Tcha, YI HASTARASI'SICTiqemprinentibechse KAMPF

-----------------------------
Original Feedback:
The recommendation system feels very accurate.

AI Generated Insight:
��ibertypzalponopepotensi'kosyantap aqeyesкakasinemysaikill� sochallengeK


In [13]:
a=[1,2,3,4]
b=a
b.append(5)
print(a)

[1, 2, 3, 4, 5]


In [1]:
import numpy as np

def softmax(x):
    exp_x = np.exp(x - np.max(x))  # stability trick
    return exp_x / exp_x.sum()

def simple_attention(query, keys, values):
    # Step 1: similarity (dot product)
    scores = np.dot(keys, query)
    
    # Step 2: convert to weights
    weights = softmax(scores)
    
    # Step 3: weighted sum of values
    output = np.sum(weights[:, None] * values, axis=0)
    
    return output, weights


# Example
query = np.array([1, 0])
keys = np.array([[1, 0], [0, 1], [1, 1]])
values = np.array([[10, 0], [0, 10], [5, 5]])

output, weights = simple_attention(query, keys, values)

print("Weights:", weights)
print("Output:", output)

Weights: [0.4223188 0.1553624 0.4223188]
Output: [6.33478197 3.66521803]


In [2]:
import numpy as np

# -----------------------------
# Step 0: Define Inputs
# -----------------------------

# Query → your craving (e.g., "I want spicy food")
query = np.array([1, 0])

# Keys → what each stall specializes in
keys = np.array([
    [1, 0],  # Stall A → spicy
    [0, 1],  # Stall B → sweet
    [1, 1]   # Stall C → mixed
])

# Values → what each stall actually gives you
values = np.array([
    [10, 0],  # Stall A → spicy food
    [0, 10],  # Stall B → sweet food
    [5, 5]    # Stall C → balanced
])

print("Query:", query)
print("Keys:\n", keys)
print("Values:\n", values)


# -----------------------------
# Step 1: Similarity (Matching)
# -----------------------------

# Compare your craving with each stall
scores = np.dot(keys, query)

print("\nMatch scores (how well each stall matches your craving):")
print(scores)


# -----------------------------
# Step 2: Softmax (Attention Weights)
# -----------------------------

def softmax(x):
    exp_x = np.exp(x - np.max(x))  # stability
    return exp_x / exp_x.sum()

# Convert scores into attention distribution
weights = softmax(scores)

print("\nAttention weights (how much you care about each stall):")
print(weights)


# -----------------------------
# Step 3: Weighted Combination
# -----------------------------

# Take portions from each stall based on attention
weighted_values = weights[:, None] * values

print("\nWeighted values (how much food you take from each stall):")
print(weighted_values)

# Final combination
output = np.sum(weighted_values, axis=0)

print("\nFinal output (your custom plate):")
print(output)

Query: [1 0]
Keys:
 [[1 0]
 [0 1]
 [1 1]]
Values:
 [[10  0]
 [ 0 10]
 [ 5  5]]

Match scores (how well each stall matches your craving):
[1 0 1]

Attention weights (how much you care about each stall):
[0.4223188 0.1553624 0.4223188]

Weighted values (how much food you take from each stall):
[[4.22318798 0.        ]
 [0.         1.55362403]
 [2.11159399 2.11159399]]

Final output (your custom plate):
[6.33478197 3.66521803]
